# SARSA on CliffWalking-v1

This notebook demonstrates **SARSA** (State-Action-Reward-State-Action; Rummery & Niranjan, 1994), an **on-policy Temporal Difference (TD) control** algorithm.

### Mathematical Update Rule
3633
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + lpha \left[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) ight]
3633

Because SARSA updates using the actual action {t+1}$ chosen by the exploratory $\epsilon569Xgreedy policy, it factors in the risk of exploratory missteps and learns a **safer detour path**.

In [ ]:
# Install Gymnasium if needed
# !pip install gymnasium pygame matplotlib numpy

import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt


### 1. Hyperparameters & Environment Setup

In [ ]:
gamma = 0.9      # Discount factor
epsilon = 0.1    # Exploration rate
alpha = 0.5      # Learning rate
episodes = 500   # Total episodes

env = gym.make("CliffWalking-v1")
num_states = env.observation_space.n
num_actions = env.action_space.n

Q = np.zeros((num_states, num_actions))


### 2. Policy: Epsilon-Greedy

In [ ]:
def epsilon_greedy(state, eps=epsilon):
    """Epsilon-greedy action selection."""
    if random.random() < eps:
        return env.action_space.sample()
    else:
        q_s = Q[state]
        best_actions = np.where(q_s == np.max(q_s))[0]
        return np.random.choice(best_actions)


### 3. Training Loop (On-Policy SARSA)

In [ ]:
episode_rewards = []
episode_lengths = []

for episode in range(episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0
    step_count = 0

    # Initial action chosen under current policy
    action = epsilon_greedy(state)

    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_action = epsilon_greedy(next_state)

        # Correct SARSA update rule
        td_target = reward + (0 if done else gamma * Q[next_state, next_action])
        Q[state, action] += alpha * (td_target - Q[state, action])

        state = next_state
        action = next_action
        total_reward += reward
        step_count += 1

    episode_rewards.append(total_reward)
    episode_lengths.append(step_count)

    if (episode + 1) % 50 == 0:
        avg_r = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1:3d}/{episodes} | Average Reward: {avg_r:.1f}")

env.close()


### 4. Evaluation (Greedy Path)

In [ ]:
test_env = gym.make("CliffWalking-v1")
state, _ = test_env.reset()
done = False
total_reward = 0
path = [state]

while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = test_env.step(action)
    done = terminated or truncated
    path.append(state)
    total_reward += reward

print(f"Evaluation Total Reward: {total_reward}")
print(f"Evaluation Path Length: {len(path) - 1} steps")
print(f"Path Visited: {path}")
test_env.close()
